# PDE: change a prompt, watch a microwave rollout
Compare **“close the microwave”** with an instruction you write. The VLA weights,
initial simulator state, and inference seed stay fixed. Then ask a VLM to describe
the reference rollout and propose new instructions, and try one of them.

Use **Runtime → Change runtime type → GPU**. This notebook targets **one A100 or L4**
(native BF16), with at least **32 GB host RAM** recommended for model loading and
**40 GB free disk** for the environment and checkpoint. A free T4 runtime is not
supported by this BF16 path. Colab GPU availability varies.

The default is [RLinf’s public pi0.5 LIBERO SFT checkpoint](https://huggingface.co/RLinf/RLinf-Pi05-LIBERO-SFT),
not the paper's weak checkpoint. This is an interactive illustration; a reworded
prompt is not guaranteed to perform better. You can supply your own compatible
checkpoint. Rollouts use one environment and frozen-policy inference, without
starting a distributed RL job. The optional VLM uses OpenAI or a Qwen endpoint
hosted elsewhere, leaving the Colab GPU for the VLA.

**Validation:** the installer’s Python environment, real pi0.5 rollouts, Qwen
feedback, suggestion replay and video display code were tested on one H100 GPU.
Google-hosted Colab, its system-package installation and live OpenAI calls were
not exercised. See [recorded validation results](https://github.com/xinyunsunshine/PDE/blob/main/notebooks/microwave_validation.json).
Setup may take several minutes and downloads several GB.


In [ ]:
#@title 1. Check the GPU and clone PDE
import os
import sys
import json
import subprocess
from pathlib import Path

subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
capabilities = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True)
if float(capabilities.strip().splitlines()[0]) < 8.0:
    raise RuntimeError("Choose an A100 or L4 runtime before installing: this pi0.5 path uses native BF16")
REPO = Path("/content/PDE")
if not REPO.exists():
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/xinyunsunshine/PDE.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=REPO, check=True)
print("PDE revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
#@title 2. Install the isolated Python 3.11 inference environment
# Keep Colab's notebook kernel unchanged. No kernel restart is needed.
subprocess.run(["bash", "scripts/install_colab.sh"], cwd=REPO, check=True)
PYTHON = "/content/pde-env/bin/python"
os.environ.update({"MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl",
                   "JAX_PLATFORMS": "cpu", "LIBERO_TYPE": "standard"})

def run_demo(*args):
    subprocess.run([PYTHON, "-m", "pde.demo", *map(str, args)], cwd=REPO, check=True)

subprocess.run([PYTHON, "-c", "import torch; print(torch.cuda.get_device_name()); assert torch.cuda.get_device_capability()[0] >= 8, 'Choose an A100 or L4 runtime'"], check=True)


In [ ]:
#@title 3. Choose and download the VLA checkpoint
checkpoint_id = "RLinf/RLinf-Pi05-LIBERO-SFT" #@param {type:"string"}
checkpoint_revision = "main" #@param {type:"string"}
# A compatible local checkpoint directory can be supplied instead of downloading.
local_checkpoint = "" #@param {type:"string"}
CHECKPOINT = local_checkpoint or "/content/pde-checkpoint"
if not local_checkpoint:
    script = """from huggingface_hub import HfApi, snapshot_download
from pathlib import Path
import json, sys
repo, revision, destination = sys.argv[1:]
sha = HfApi().model_info(repo, revision=revision, token=False).sha
snapshot_download(repo_id=repo, revision=sha, local_dir=destination, token=False)
Path(destination, 'demo_source.json').write_text(json.dumps({'repo_id': repo, 'revision': sha}))
print('Checkpoint revision:', sha)
"""
    subprocess.run([PYTHON, "-c", script, checkpoint_id, checkpoint_revision, CHECKPOINT], check=True)
print("Using checkpoint:", CHECKPOINT)


## Compare two prompts
Edit `custom_prompt` and run the next two cells again. Each comparison loads the
policy once, runs both prompts sequentially, and releases GPU memory when done.
`trial` selects the same benchmark initial state for both rollouts; `seed` fixes
the policy's random noise. Success is measured by the simulator's original
microwave task condition, regardless of the instruction supplied to the VLA.


In [ ]:
#@title 4. Write a prompt and run the comparison
custom_prompt = "push the microwave door until it shuts" #@param {type:"string"}
seed = 0 #@param {type:"integer"}
trial = 0 #@param {type:"integer"}
steps = 240 #@param {type:"integer"}
OUTPUT = Path("/content/microwave-demo")
run_demo("compare", "--checkpoint", CHECKPOINT, "--prompt", custom_prompt,
         "--seed", seed, "--trial", trial, "--steps", steps, "--output", OUTPUT)


In [ ]:
#@title 5. Watch both behaviors side by side
from IPython.display import HTML, display
import base64
import html

def show_comparison(directory):
    report = json.loads((directory / "comparison.json").read_text())
    panels = []
    for result in report["results"]:
        data = base64.b64encode((directory / result["video"]).read_bytes()).decode()
        panels.append(f"<div style='flex:1;min-width:240px'><h4>{html.escape(result['prompt'])}</h4>"
                      f"<p>Simulator success: {result['success']}</p>"
                      f"<video controls loop style='width:100%' src='data:video/mp4;base64,{data}'></video></div>")
    display(HTML("<div style='display:flex;gap:20px;flex-wrap:wrap'>" + "".join(panels) + "</div>"))
    print("Same initial image:", report["results"][0]["initial_image_sha256"] == report["results"][1]["initial_image_sha256"])

show_comparison(OUTPUT)


## Ask a VLM what happened
This optional step makes **two API requests**: one with eight frames from the
reference rollout to summarize behavior, then one to propose three prompts from
that summary. It displays the **actual returned text**, not a saved or fabricated
example. An OpenAI call sends the sampled frames to OpenAI and incurs API usage.
Choose `local_qwen` for your own reachable OpenAI-compatible Qwen server; it should
run on another machine so the Colab GPU remains available for the VLA.

Enter the key through the hidden input or Colab Secrets (`OPENAI_API_KEY` or
`QWEN_API_KEY`). Keys are not written into the notebook or result files.


### Recorded example from a real VLM call
The following is a **recorded Qwen3-VL-2B-Instruct response** from our H100 test,
using eight frames from the canonical rollout (seed 0, trial 0). Your live call
below produces a new response. The summary is the VLM’s interpretation;
simulator success was **false** for the original, manual rewrite and first VLM
suggestion in this test. This is an execution check, not evidence of an improvement.

```json
{
  "provider": "local_qwen",
  "model": "Qwen/Qwen3-VL-2B-Instruct",
  "summary": "The robot attempted to close the microwave by placing a mug inside it, but failed to do so as it was not properly aligned and the microwave door did not close correctly.",
  "new_prompts": [
    "close the microwave door properly",
    "ensure the microwave door is fully closed",
    "close the microwave without a mug inside"
  ],
  "simulator_success": false
}
```


In [ ]:
#@title 6. Call the VLM and display its output
provider = "openai" #@param ["openai", "local_qwen"]
vlm_model = "gpt-4.1" #@param {type:"string"}
qwen_base_url = "" #@param {type:"string"}
# For local_qwen, set the server's model ID above (e.g. Qwen/Qwen3-VL-8B-Instruct).
from getpass import getpass
from google.colab import userdata
key_name = "OPENAI_API_KEY" if provider == "openai" else "QWEN_API_KEY"
try:
    api_key = userdata.get(key_name)
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    api_key = getpass(f"{key_name} (local Qwen without authentication: leave blank): ")
if provider == "openai" and not api_key:
    raise ValueError("An OpenAI API key is required for this optional step")
if provider == "local_qwen" and not qwen_base_url:
    raise ValueError("Enter the reachable Qwen server URL, including /v1")
args = ["feedback", "--output", OUTPUT, "--provider", provider, "--model", vlm_model]
if provider == "local_qwen":
    args += ["--base-url", qwen_base_url]
previous_key = os.environ.get(key_name)
try:
    if api_key:
        os.environ[key_name] = api_key
    else:
        os.environ.pop(key_name, None)
    run_demo(*args)
finally:
    if previous_key is None:
        os.environ.pop(key_name, None)
    else:
        os.environ[key_name] = previous_key
    del api_key
feedback = json.loads((OUTPUT / "vlm_feedback.json").read_text())
print("VLM summary:\n", feedback["summary"])
print("Suggested prompts:")
for i, prompt in enumerate(feedback["new_prompts"]):
    print(f"{i}: {prompt}")
print("Simulator success (ground truth):", feedback["simulator_success"])


In [ ]:
#@title 7. Try a VLM suggestion against the original prompt
suggestion_index = 0 #@param {type:"integer"}
if not 0 <= suggestion_index < len(feedback["new_prompts"]):
    raise ValueError("Choose an index printed in the previous cell")
suggested_prompt = feedback["new_prompts"][suggestion_index]
VLM_OUTPUT = Path("/content/microwave-vlm-demo")
run_demo("compare", "--checkpoint", CHECKPOINT, "--prompt", suggested_prompt,
         "--seed", seed, "--trial", trial, "--steps", steps, "--output", VLM_OUTPUT)
show_comparison(VLM_OUTPUT)


## Keep your results
Videos, sampled frames, prompt text, simulator success and seed are saved under
`/content/microwave-demo/` (and `/content/microwave-vlm-demo/` for the suggested
prompt). Download them before disconnecting; Colab runtime storage is temporary.
A single matched pair illustrates behavior and is not a success-rate estimate.
If setup or inference fails, keep the full traceback and the package versions in
`/content/pde-env/installed-requirements.txt`. For CUDA memory errors, choose a
larger GPU; for host RAM exhaustion while loading, choose a high-memory runtime.

[Code](https://github.com/xinyunsunshine/PDE) ·
[RLinf](https://github.com/RLinf/RLinf) ·
[Colab runtime guidance](https://research.google.com/colaboratory/faq.html)
